<a href="https://colab.research.google.com/github/gitansh01/Redrob-AI-Campus-Hackathon/blob/main/redrob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import math
# Sample resume data
resumes = {
    "Arjun Sharma": ["Python", "MachineLearning", "SQL", "pandas", "numpy", "Deep-learning"],
    "Priya Nair": ["JavaScript", "React", "Node.JS", "MongoDb", "REST api", "HTML/CSS"],
    "Rahul Gupta": ["Java", "Spring Boot", "MySql", "Microservices", "Docker", "kubernates"],
    "Sneha Patel": ["Python", "TensorFlow", "Keras", "NLP", "BERT", "data-viz", "matplotlib"],
    "Vikram": []
}

# Sample job description data
jd = ["Python", "MachineLearning", "SQL", "pandas", "numpy"]

def normalize_skills(skills_list):
    """Normalize the skills in a list by converting to lowercase and removing special characters"""
    normalized_skills = []
    for skill in skills_list:
        normalized_skill = ''.join(e for e in skill if e.isalnum()).lower()
        normalized_skills.append(normalized_skill)
    return normalized_skills

def deduplicate_skills(skills_list):
    """Deduplicate the skills in a list"""
    unique_skills = set(skills_list)
    return list(unique_skills)

def calculate_tf_idf(resume_skills, global_skill_document_frequency, num_total_resumes):
    """Calculate the TF-IDF vector for a resume"""
    tf_idf_vector = {}
    if not resume_skills: # Handle empty resumes
        return tf_idf_vector

    # Term Frequency (TF)
    skill_counts = {}
    for skill in resume_skills:
        skill_counts[skill] = skill_counts.get(skill, 0) + 1

    for skill, count in skill_counts.items():
        tf = count / len(resume_skills) # Raw count / total skills in resume

        # Inverse Document Frequency (IDF)
        df = global_skill_document_frequency.get(skill, 0)
        if df > 0:
            idf = math.log(num_total_resumes / df)
        else:
            idf = 0 # Skill not found in any resume in the corpus

        tf_idf_vector[skill] = tf * idf
    return tf_idf_vector

def create_jd_binary_vector(jd_skills, global_all_unique_skills):
    """Create a binary vector for a job description based on the global unique skills"""
    binary_vector = {}
    normalized_jd_skills = normalize_skills(jd_skills)
    for skill in global_all_unique_skills:
        if skill in normalized_jd_skills:
            binary_vector[skill] = 1
        else:
            binary_vector[skill] = 0
    return binary_vector

def calculate_cosine_similarity(resume_vector, jd_vector):
    """Calculate the cosine similarity between a resume vector and a JD vector"""
    # Ensure both vectors have the same set of skills, filling missing with 0
    all_skills = set(resume_vector.keys()).union(set(jd_vector.keys()))

    dot_product = 0
    magnitude_resume = 0
    magnitude_jd = 0

    for skill in all_skills:
        res_val = resume_vector.get(skill, 0)
        jd_val = jd_vector.get(skill, 0)
        dot_product += res_val * jd_val
        magnitude_resume += res_val ** 2
        magnitude_jd += jd_val ** 2

    magnitude_resume = math.sqrt(magnitude_resume)
    magnitude_jd = math.sqrt(magnitude_jd)

    if magnitude_resume == 0 or magnitude_jd == 0:
        return 0.0 # Avoid division by zero if one vector is all zeros

    cosine_similarity = dot_product / (magnitude_resume * magnitude_jd)
    return cosine_similarity

def rank_resumes(resumes_data, jd_data):
    """Rank the resumes based on their cosine similarity scores"""

    # Step 1: Pre-process all resumes to get global unique skills and document frequencies
    all_processed_resume_skills = {}
    global_all_unique_skills = set()
    global_skill_document_frequency = {}
    num_total_resumes = len(resumes_data)

    for resume_name, skills_list in resumes_data.items():
        normalized_skills = normalize_skills(skills_list)
        unique_skills = deduplicate_skills(normalized_skills)
        all_processed_resume_skills[resume_name] = unique_skills
        global_all_unique_skills.update(unique_skills)

    for skill in global_all_unique_skills:
        count = 0
        for resume_name, unique_skills in all_processed_resume_skills.items():
            if skill in unique_skills:
                count += 1
        global_skill_document_frequency[skill] = count

    # Normalize and deduplicate JD skills once
    normalized_jd_data = normalize_skills(jd_data)
    unique_jd_skills = deduplicate_skills(normalized_jd_data)

    ranked_resumes = []
    for resume_name, skills_list in resumes_data.items():
        processed_skills_for_current_resume = all_processed_resume_skills[resume_name]

        tf_idf_vector = calculate_tf_idf(processed_skills_for_current_resume, global_skill_document_frequency, num_total_resumes)

        # Create JD binary vector using the global set of unique skills
        jd_binary_vector = create_jd_binary_vector(unique_jd_skills, global_all_unique_skills)

        cosine_similarity = calculate_cosine_similarity(tf_idf_vector, jd_binary_vector)
        ranked_resumes.append((resume_name, cosine_similarity))

    ranked_resumes.sort(key=lambda x: x[1], reverse=True)
    return ranked_resumes

# Test the code
ranked_resumes = rank_resumes(resumes, jd)
for resume, score in ranked_resumes:
    print(f"{resume}: {score}")

Arjun Sharma: 0.8856101931878053
Sneha Patel: 0.10124502636758939
Priya Nair: 0.0
Rahul Gupta: 0.0
Vikram: 0.0
